# Deep Learning Text-to-SQL Model from Scratch
### Coursework: Semester 5 Deep Learning Project

This notebook demonstrates building and training a **Sequence-to-Sequence (Seq2Seq) Transformer Model** from scratch in PyTorch to translate natural language queries into valid SQL, trained on the **Gretel Synthetic Text-to-SQL** dataset.

### Pipeline Overview:
1. **Dataset Ingestion & Schema Conditioning**: Extract question + DDL schema context (`gretelai/synthetic_text_to_sql`).
2. **Custom SQL-Aware Tokenization**: Regex-based tokenization preserving SQL syntax, symbols, and keywords.
3. **Transformer Architecture from Scratch**: Positional Encoding, Multi-Head Attention, Pre-LN Encoder/Decoder stacks.
4. **Training with Teacher Forcing**: CrossEntropy loss with label smoothing and AdamW optimizer.
5. **Inference & Evaluation**: Greedy & Beam Search decoding + SQLite syntax validation.

In [ ]:
import os
import json
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.dataset import prepare_and_cache_dataset, Text2SQLDataset, collate_fn
from src.tokenizer import SQLTokenizer
from src.models import TransformerSeq2Seq, LSTMSeq2Seq
from src.train import train_model
from src.inference import predict_sql, validate_sql_syntax
from src.evaluate import evaluate_model_on_test_set

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device.upper()}")

## 1. Data Preparation & Caching
We fetch and cache a subset of the dataset and clean the database schema (extracting `CREATE TABLE` DDLs).

In [ ]:
train_path, val_path = prepare_and_cache_dataset(output_dir="data", num_samples=3000, val_split=0.1)

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(val_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"Loaded {len(train_data)} training and {len(val_data)} validation samples.")
print("Sample training instance:")
print("Input:", train_data[0]["src"])
print("Target SQL:", train_data[0]["tgt"])

## 2. Tokenizer & Vocabulary Construction
We fit our custom tokenizer on the natural language questions + schema and target SQL.

In [ ]:
src_tokenizer = SQLTokenizer(min_freq=2, max_vocab_size=8000)
src_tokenizer.build_vocab([d["src"] for d in train_data])

tgt_tokenizer = SQLTokenizer(min_freq=2, max_vocab_size=8000)
tgt_tokenizer.build_vocab([d["tgt"] for d in train_data])

print(f"Source Vocab Size: {len(src_tokenizer):,}")
print(f"Target Vocab Size: {len(tgt_tokenizer):,}")

## 3. Model Initialization (Transformer from Scratch)
We build an end-to-end Seq2Seq Transformer model using multi-head self-attention and cross-attention.

In [ ]:
model = TransformerSeq2Seq(
    src_vocab_size=len(src_tokenizer),
    tgt_vocab_size=len(tgt_tokenizer),
    d_model=256,
    num_heads=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    d_ff=512,
    dropout=0.1,
    pad_idx=src_tokenizer.PAD_IDX
)

print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Training Loop

In [ ]:
train_dataset = Text2SQLDataset(train_data, src_tokenizer, tgt_tokenizer)
val_dataset = Text2SQLDataset(val_data, src_tokenizer, tgt_tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=lambda b: collate_fn(b, src_tokenizer.PAD_IDX))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=lambda b: collate_fn(b, src_tokenizer.PAD_IDX))

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=5e-4,
    save_dir="checkpoints",
    device=device
)

## 5. Loss Curves & Performance Plots

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.title("Cross-Entropy Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history["val_acc"], label="Val Token Accuracy", color="green")
plt.title("Validation Token Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. Inference & Interactive Testing

In [ ]:
test_question = "What is the total revenue per department?"
test_schema = "CREATE TABLE sales (id INT, department TEXT, revenue REAL);"

pred_sql = predict_sql(model, test_question, test_schema, src_tokenizer, tgt_tokenizer, use_beam_search=True, device=device)
valid, err = validate_sql_syntax(pred_sql)

print("Question: ", test_question)
print("Schema:   ", test_schema)
print("Predicted SQL: ", pred_sql)
print("Syntax Valid:", valid)